# LeetCode 3943 — Number of Pairs After Increment bằng Python

- Link: https://leetcode.com/problems/number-of-pairs-after-increment/
- Độ khó: **Hard**
- Chủ đề: Phân rã căn bậc hai (sqrt decomposition), cộng lười (lazy) theo khối, bảng đếm `dict`/`Counter`, vectorize bằng NumPy

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numberOfPairs(self, nums1: list[int], nums2: list[int], queries: list[list[int]]) -> list[int]:
```

## 1. Giới thiệu & mục tiêu học

Có hai mảng `nums1` (rất ngắn, tối đa 5 phần tử) và `nums2` (tới `5·10^4`), cùng hai loại truy vấn xen kẽ:

- `[1, x, y, val]`: cộng `val` vào mọi phần tử `nums2[x..y]`;
- `[2, tot]`: đếm số cặp `(j, k)` thỏa `nums1[j] + nums2[k] == tot`.

Vì `nums1` ngắn, mỗi truy vấn loại 2 chỉ cần **tối đa 5 câu hỏi** dạng "trong `nums2` hiện tại có bao nhiêu phần tử bằng `tot − a`?". Phần khó là trả lời câu hỏi đó khi `nums2` liên tục bị **cộng cả đoạn**.

Sau notebook này bạn sẽ:

1. Rút gọn bài về "đếm số phần tử bằng `t` sau các phép cộng đoạn".
2. Hiểu **phân rã căn bậc hai**: chia mảng thành khối, mỗi khối có bảng đếm + một **tag cộng lười** `lazy`.
3. Biết xử lý **khối đầy** (chỉ cộng `lazy`) và **khối lẻ ở hai đầu** (sửa từng phần tử + sửa bảng đếm).
4. Biết vì sao truy vấn đếm tra `cnt.get(t − lazy)` chứ không phải `cnt.get(t)`.
5. So sánh 3 cách: vét cạn, phân rã căn bậc hai (`O((n + q)·√n)`), và vét cạn **vectorize bằng NumPy** (nhanh nhờ hằng số cực nhỏ).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums1`, `nums2` và mảng `queries`:

- `[1, x, y, val]`: `nums2[i] += val` với mọi `x ≤ i ≤ y`.
- `[2, tot]`: tính số cặp chỉ số `(j, k)` (`j` trong `nums1`, `k` trong `nums2`) sao cho `nums1[j] + nums2[k] == tot`. Hai phần tử bằng giá trị nhưng khác chỉ số vẫn là các cặp khác nhau.

Trả về mảng đáp án cho các truy vấn loại 2 theo thứ tự.

**Ví dụ 1:** `nums1 = [1,2]`, `nums2 = [3,4]`, `queries = [[2,5],[1,0,0,2],[2,5]]` → `[2,1]`.

**Ví dụ 2:** `nums1 = [1,1]`, `nums2 = [2,2,3]`, `queries = [[2,4],[1,0,1,1],[2,4]]` → `[2,6]`.

**Ví dụ 3:** `nums1 = [2,5,8,4]`, `nums2 = [1,3,8]`, `queries = [[2,9],[1,1,2,1],[2,10]]` → `[1,0]`.

**Ràng buộc:** `1 ≤ len(nums1) ≤ 5`, `1 ≤ len(nums2) ≤ 5·10^4`, `1 ≤ nums1[i], nums2[i] ≤ 10^5`, `1 ≤ len(queries) ≤ 5·10^4`, `0 ≤ x ≤ y < len(nums2)`, `1 ≤ val ≤ 10^5`, `1 ≤ tot ≤ 10^9`.

Lưu ý: `val ≥ 1` (chỉ cộng số dương). Thuật toán sqrt-decomposition dưới đây không dựa vào điều này nên vẫn đúng khi `val` âm.

In [1]:
import random, time, math
from collections import Counter

EXAMPLES = [
    (([1, 2], [3, 4], [[2, 5], [1, 0, 0, 2], [2, 5]]), [2, 1]),
    (([1, 1], [2, 2, 3], [[2, 4], [1, 0, 1, 1], [2, 4]]), [2, 6]),
    (([2, 5, 8, 4], [1, 3, 8], [[2, 9], [1, 1, 2, 1], [2, 10]]), [1, 0]),
]
for args, exp in EXAMPLES:
    print("nums1 =", args[0], "nums2 =", args[1], "-> kỳ vọng", exp)

nums1 = [1, 2] nums2 = [3, 4] -> kỳ vọng [2, 1]
nums1 = [1, 1] nums2 = [2, 2, 3] -> kỳ vọng [2, 6]
nums1 = [2, 5, 8, 4] nums2 = [1, 3, 8] -> kỳ vọng [1, 0]


## 3. Kiến thức nền

### 3.1. `Counter` / `dict.get` — bảng đếm

`Counter(a)` đếm số lần mỗi giá trị xuất hiện. `cnt.get(v, 0)` trả `0` nếu `v` chưa có (không báo lỗi, không tạo khóa mới). Tra cứu trung bình `O(1)`.

In [2]:
c = Counter([3, 4, 3, 8])
print(c)
print(c.get(3, 0), c.get(5, 0))      # 2 0
c[3] -= 1; c[4] += 5                 # sửa bảng đếm khi một phần tử đổi giá trị
print(c)

Counter({3: 2, 4: 1, 8: 1})
2 0
Counter({4: 6, 3: 1, 8: 1})


### 3.2. Chia khối và chỉ số khối

Chia mảng độ dài `n` thành các khối kích thước `B`: phần tử `i` thuộc khối `i // B`; khối `b` gồm các chỉ số `[b·B, (b+1)·B)` (khối cuối có thể ngắn). Số khối `nb = ceil(n / B)`.

In [3]:
n, B = 10, 4
print([i // B for i in range(n)])            # [0,0,0,0,1,1,1,1,2,2]
print("số khối:", (n + B - 1) // B)
for b in range((n + B - 1) // B):
    print("khối", b, "gồm chỉ số", list(range(b * B, min(n, (b + 1) * B))))

[0, 0, 0, 0, 1, 1, 1, 1, 2, 2]
số khối: 3
khối 0 gồm chỉ số [0, 1, 2, 3]
khối 1 gồm chỉ số [4, 5, 6, 7]
khối 2 gồm chỉ số [8, 9]


### 3.3. Cộng lười (lazy) theo khối

Muốn cộng `v` cho **cả khối** mà không duyệt từng phần tử: ghi nhớ `lazy[b] += v` và coi **giá trị thật** của phần tử `i` trong khối `b` là `base[i] + lazy[b]` (`base` là giá trị đang lưu). Bảng đếm của khối lưu **số lần xuất hiện của `base`**, nên khi hỏi "có bao nhiêu phần tử **thật** bằng `t`" ta tra `cnt.get(t − lazy[b])`.

In [4]:
base = [1, 3, 3, 8]                  # một khối
cnt = Counter(base)
lazy = 0
lazy += 5                            # cộng 5 cho cả khối: giá trị thật là [6, 8, 8, 13]
print("giá trị thật:", [x + lazy for x in base])
t = 8
print("số phần tử thật bằng 8:", cnt.get(t - lazy, 0), "(tra cnt.get(3))")

giá trị thật: [6, 8, 8, 13]
số phần tử thật bằng 8: 2 (tra cnt.get(3))


### 3.4. NumPy: cộng lát cắt và đếm bằng `count_nonzero`

`arr[x:y+1] += v` cộng cả đoạn bằng một lệnh vector (chạy bằng C, theo từng khối bộ nhớ). `np.count_nonzero(arr == t)` đếm số phần tử bằng `t`. Mỗi lệnh duyệt toàn mảng nhưng với hằng số rất nhỏ (cỡ `10^-10` giây mỗi phần tử nhờ SIMD).

In [5]:
import numpy as np
arr = np.array([1, 3, 3, 8], dtype=np.int64)
arr[1:3] += 5
print(arr, np.count_nonzero(arr == 8))

[1 8 8 8] 3


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `nums1 = [1, 2]`, `nums2 = [3, 4]`.

- Truy vấn `[2, 5]`: cặp `(1, 4)` và `(2, 3)` → `2`.
- Cộng `2` vào `nums2[0]`: `nums2 = [5, 4]`.
- Truy vấn `[2, 5]`: cần `nums2` bằng `5 − 1 = 4` hoặc `5 − 2 = 3`: chỉ có `4` → `1`.

Vậy một truy vấn loại 2 = `Σ_{a ∈ nums1} đếm(nums2 == tot − a)`. Việc còn lại là cấu trúc dữ liệu cho `đếm(nums2 == t)` có hỗ trợ cộng đoạn.

Cách đơn giản: giữ nguyên `nums2` và quét hết mỗi lần. `nums1` ngắn nên chỉ có 5 lần quét/ truy vấn — nhưng mỗi lần quét `O(n)`.

In [6]:
def count_pairs_once(nums1, nums2, tot):
    c = Counter(nums2)
    return sum(c.get(tot - a, 0) for a in nums1)

print(count_pairs_once([1, 2], [3, 4], 5), count_pairs_once([1, 2], [5, 4], 5))

2 1


## 5. Cách 1 — Brute Force

**Ý tưởng.** Giữ `nums2` thật; mỗi truy vấn loại 1 cộng từng phần tử trong đoạn; mỗi truy vấn loại 2 dựng `Counter(nums2)` rồi cộng `Counter.get(tot − a)` cho `a` trong `nums1`.

In [7]:
def brute(nums1, nums2, queries):
    arr = list(nums2)
    out = []
    for q in queries:
        if q[0] == 1:
            _, x, y, val = q
            for i in range(x, y + 1):
                arr[i] += val
        else:
            tot = q[1]
            c = Counter(arr)
            out.append(sum(c.get(tot - a, 0) for a in nums1))
    return out

for args, exp in EXAMPLES:
    got = brute(*args)
    print(got, "(kỳ vọng", exp, ")")
    assert got == exp

[2, 1] (kỳ vọng [2, 1] )
[2, 6] (kỳ vọng [2, 6] )
[1, 0] (kỳ vọng [1, 0] )


**Độ phức tạp.** Mỗi truy vấn tốn `O(n)` → tổng `O(q·n) = 2.5·10^9` với giới hạn lớn nhất. Quá chậm trong Python thuần.

**Nút thắt.** Truy vấn loại 1 cộng lại từng phần tử và truy vấn loại 2 dựng lại bảng đếm từ đầu. Cần một cấu trúc **giữ sẵn** bảng đếm và cho phép cộng đoạn gần `O(√n)`.

## 6. Bước đột phá

**Phân rã căn bậc hai.** Chia `nums2` thành các khối kích thước `B ≈ √n`. Mỗi khối `b` có:

- `base`: giá trị lưu cho từng phần tử (chưa tính tag của khối);
- `lazy[b]`: tag cộng của cả khối;
- `cnt[b]`: bảng đếm của các `base` trong khối.

Giá trị thật của phần tử = `base[i] + lazy[block(i)]`.

**Truy vấn cộng `[x, y]`, `val`:**

- **Khối nằm hoàn toàn trong `[x, y]`:** `lazy[b] += val` — `O(1)`, bảng đếm giữ nguyên (mọi `base` dịch cùng nhau).
- **Khối chỉ bị phủ một phần (tối đa hai khối ở hai đầu):** với từng phần tử bị phủ: `cnt[b][base[i]] −= 1`, `base[i] += val`, `cnt[b][base[i]] += 1` — `O(B)`.
- Nếu `x` và `y` cùng một khối: chỉ xử lý **một lần** (đừng cộng hai nhánh).

**Truy vấn đếm `tot`:** với mỗi `a ∈ nums1` (nên gom các `a` trùng nhau thành `a → số lần` để khỏi lặp), `t = tot − a`, rồi cộng qua mọi khối `cnt[b].get(t − lazy[b], 0)`: `O(n / B)` cho mỗi `a`.

**Tổng chi phí:** cộng `O(B + n/B)`, đếm `O(|A| · n/B)` với `|A| ≤ 5`. Chọn `B ≈ √(5n)` để cân bằng ≈ `O(√(5n))` mỗi truy vấn: với `n = 5·10^4` là cỡ `~500` thao tác.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Phân rã căn bậc hai với `Counter` và `lazy`

**Giải thích từng đoạn.**

1. `B = max(1, int(sqrt(5n)))`; `nb = ceil(n / B)`; `base = nums2[:]`; `cnt[b] = Counter(base[b·B:(b+1)·B])`; `lazy = [0] * nb`.
2. Hàm phụ `apply_partial(b, l, r, val)` sửa phần tử `l..r` (thuộc khối `b`).
3. Truy vấn loại 1: tìm `bx = x // B`, `by = y // B`. Nếu `bx == by`: `apply_partial(bx, x, y)`. Ngược lại: `apply_partial(bx, x, (bx+1)·B − 1)`, các khối ở giữa `lazy += val`, `apply_partial(by, by·B, y)`.
4. Truy vấn loại 2: như mục 6.

In [8]:
def sqrt_decomp(nums1, nums2, queries, B=None):
    n = len(nums2)
    if B is None:
        B = max(1, int(math.sqrt(5 * n)))
    nb = (n + B - 1) // B
    base = list(nums2)
    lazy = [0] * nb
    cnt = [Counter(base[b * B:(b + 1) * B]) for b in range(nb)]
    groups = list(Counter(nums1).items())            # (giá trị a, số lần xuất hiện)

    def apply_partial(b, l, r, val):
        c = cnt[b]
        for i in range(l, r + 1):
            old = base[i]
            c[old] -= 1
            base[i] = old + val
            c[old + val] += 1

    out = []
    for q in queries:
        if q[0] == 1:
            _, x, y, val = q
            bx, by = x // B, y // B
            if bx == by:
                apply_partial(bx, x, y, val)
            else:
                apply_partial(bx, x, (bx + 1) * B - 1, val)
                for b in range(bx + 1, by):
                    lazy[b] += val
                apply_partial(by, by * B, y, val)
        else:
            tot = q[1]
            ans = 0
            for a, mult in groups:
                t = tot - a
                s = 0
                for c, lz in zip(cnt, lazy):
                    s += c.get(t - lz, 0)
                ans += mult * s
            out.append(ans)
    return out

for args, exp in EXAMPLES:
    got = sqrt_decomp(*args)
    print(got, "(kỳ vọng", exp, ")")
    assert got == exp

[2, 1] (kỳ vọng [2, 1] )
[2, 6] (kỳ vọng [2, 6] )
[1, 0] (kỳ vọng [1, 0] )


**Độ phức tạp:** thời gian `O(n + q · (B + |A|·n/B))` ⇒ với `B ≈ √(5n)`: `O(n + q·√(5n)·c)`; không gian `O(n)`.

Trong Python thuần, hằng số của `dict` và vòng lặp khiến trường hợp tồi nhất (`n = q = 5·10^4`) cỡ vài chục giây ở máy yếu, khoảng 10 giây ở máy hiện đại. Đó là lý do ta xét Cách 3.

### 7.2. Cách 3 — Vét cạn vectorize bằng NumPy

**Giải thích.** Về lý thuyết vẫn `O(q · n · |A|)` nhưng:

- cộng đoạn `arr[x:y+1] += val` là một lệnh C (bộ nhớ liên tục);
- `np.count_nonzero(arr == t)` duyệt `5·10^4` số nguyên 64-bit trong khoảng chục micro giây nhờ SIMD.

Tổng khoảng `q·5·10 µs ≈ 2.5 s` ở tải tối đa (thực đo gần 2 giây). Vì chi phí **không phụ thuộc dữ liệu** nên không có "ca xấu" bất ngờ. Dùng được khi LeetCode có NumPy (thường có).

In [9]:
import numpy as np

def numpy_brute(nums1, nums2, queries):
    arr = np.array(nums2, dtype=np.int64)
    out = []
    for q in queries:
        if q[0] == 1:
            arr[q[1]:q[2] + 1] += q[3]
        else:
            tot = q[1]
            out.append(sum(int(np.count_nonzero(arr == tot - a)) for a in nums1))
    return out

for args, exp in EXAMPLES:
    got = numpy_brute(*args)
    print(got, "(kỳ vọng", exp, ")")
    assert got == exp

[2, 1] (kỳ vọng [2, 1] )
[2, 6] (kỳ vọng [2, 6] )
[1, 0] (kỳ vọng [1, 0] )


**Độ phức tạp:** `O(q · n · |A|)` thao tác với hằng số cực nhỏ; không gian `O(n)`.

**Chọn cách nào để nộp?** Cách 2 là thuật toán có **bảo đảm tiệm cận** (`~√n` mỗi truy vấn) và không cần thư viện ngoài; Cách 3 thực tế nhanh hơn nhiều trong Python. Ở mục 9 ta đưa **cả hai** vào `Solution` (lớp chính dùng Cách 2) và `SolutionNumpy` (dự phòng).

## 8. TRACE từng bước

Trace Cách 2 với `nums1 = [2, 5, 8, 4]`, `nums2 = [1, 3, 8]`, `B = 2` (hai khối: `[1, 3]` và `[8]`) và các truy vấn của Ví dụ 3.

| Bước | Hành động | `base` | `lazy` | `cnt` |
|---|---|---|---|---|
| đầu | khởi tạo | `[1,3,8]` | `[0,0]` | `{1:1,3:1}`, `{8:1}` |
| `[2,9]` | đếm `t = 9 − a` | | | `t = 7, 4, 1, 5` → chỉ `t = 1` có (khối 0) → **1** |
| `[1,1,2,1]` | `x = 1` (khối 0), `y = 2` (khối 1): phủ một phần khối 0 (phần tử 1) và cả khối 1 (nhưng khối 1 chỉ có phần tử 2, đầu mút `y` ở khối 1 → xử lý như phần lẻ) | `[1,4,9]` | `[0,0]` | `{1:1,4:1}`, `{9:1}` |
| `[2,10]` | `t = 8, 5, 2, 6` | | | không có → **0** |

Chạy hàm trace để xem từng bước của chương trình.

In [10]:
def trace_sqrt(nums1, nums2, queries, B):
    n = len(nums2)
    nb = (n + B - 1) // B
    base = list(nums2)
    lazy = [0] * nb
    cnt = [Counter(base[b * B:(b + 1) * B]) for b in range(nb)]
    groups = list(Counter(nums1).items())
    show = lambda: print(f"      base={base}  lazy={lazy}  cnt={[dict((k, v) for k, v in c.items() if v) for c in cnt]}")
    print("khởi tạo:"); show()
    out = []
    for qi, q in enumerate(queries):
        print(f"\ntruy vấn {qi}: {q}")
        if q[0] == 1:
            _, x, y, val = q
            bx, by = x // B, y // B
            print(f"   khối của x = {bx}, của y = {by}")
            def part(b, l, r):
                print(f"   sửa từng phần tử khối {b}, chỉ số {l}..{r}")
                c = cnt[b]
                for i in range(l, r + 1):
                    old = base[i]; c[old] -= 1; base[i] = old + val; c[old + val] += 1
            if bx == by:
                part(bx, x, y)
            else:
                part(bx, x, (bx + 1) * B - 1)
                for b in range(bx + 1, by):
                    lazy[b] += val
                    print(f"   khối đầy {b}: lazy += {val}")
                part(by, by * B, y)
            show()
        else:
            tot = q[1]
            ans = 0
            for a, mult in groups:
                t = tot - a
                found = [c.get(t - lz, 0) for c, lz in zip(cnt, lazy)]
                print(f"   a={a}(x{mult}) -> t={t}, mỗi khối tra cnt.get(t - lazy): {found}")
                ans += mult * sum(found)
            print("   đáp án:", ans)
            out.append(ans)
    return out

args = EXAMPLES[2][0]
assert trace_sqrt(*args, B=2) == EXAMPLES[2][1]

khởi tạo:
      base=[1, 3, 8]  lazy=[0, 0]  cnt=[{1: 1, 3: 1}, {8: 1}]

truy vấn 0: [2, 9]
   a=2(x1) -> t=7, mỗi khối tra cnt.get(t - lazy): [0, 0]
   a=5(x1) -> t=4, mỗi khối tra cnt.get(t - lazy): [0, 0]
   a=8(x1) -> t=1, mỗi khối tra cnt.get(t - lazy): [1, 0]
   a=4(x1) -> t=5, mỗi khối tra cnt.get(t - lazy): [0, 0]
   đáp án: 1

truy vấn 1: [1, 1, 2, 1]
   khối của x = 0, của y = 1
   sửa từng phần tử khối 0, chỉ số 1..1
   sửa từng phần tử khối 1, chỉ số 2..2
      base=[1, 4, 9]  lazy=[0, 0]  cnt=[{1: 1, 4: 1}, {9: 1}]

truy vấn 2: [2, 10]
   a=2(x1) -> t=8, mỗi khối tra cnt.get(t - lazy): [0, 0]
   a=5(x1) -> t=5, mỗi khối tra cnt.get(t - lazy): [0, 0]
   a=8(x1) -> t=2, mỗi khối tra cnt.get(t - lazy): [0, 0]
   a=4(x1) -> t=6, mỗi khối tra cnt.get(t - lazy): [0, 0]
   đáp án: 0


In [11]:
# Một trace có khối đầy (lazy): nums2 có 6 phần tử, B = 2
out = trace_sqrt([1, 3], [2, 2, 5, 5, 7, 8], [[2, 6], [1, 0, 5, 1], [2, 6], [1, 1, 4, 2], [2, 6]], B=2)
assert out == brute([1, 3], [2, 2, 5, 5, 7, 8], [[2, 6], [1, 0, 5, 1], [2, 6], [1, 1, 4, 2], [2, 6]])
print("kết quả:", out)

khởi tạo:
      base=[2, 2, 5, 5, 7, 8]  lazy=[0, 0, 0]  cnt=[{2: 2}, {5: 2}, {7: 1, 8: 1}]

truy vấn 0: [2, 6]
   a=1(x1) -> t=5, mỗi khối tra cnt.get(t - lazy): [0, 2, 0]
   a=3(x1) -> t=3, mỗi khối tra cnt.get(t - lazy): [0, 0, 0]
   đáp án: 2

truy vấn 1: [1, 0, 5, 1]
   khối của x = 0, của y = 2
   sửa từng phần tử khối 0, chỉ số 0..1
   khối đầy 1: lazy += 1
   sửa từng phần tử khối 2, chỉ số 4..5
      base=[3, 3, 5, 5, 8, 9]  lazy=[0, 1, 0]  cnt=[{3: 2}, {5: 2}, {8: 1, 9: 1}]

truy vấn 2: [2, 6]
   a=1(x1) -> t=5, mỗi khối tra cnt.get(t - lazy): [0, 0, 0]
   a=3(x1) -> t=3, mỗi khối tra cnt.get(t - lazy): [2, 0, 0]
   đáp án: 2

truy vấn 3: [1, 1, 4, 2]
   khối của x = 0, của y = 2
   sửa từng phần tử khối 0, chỉ số 1..1
   khối đầy 1: lazy += 2
   sửa từng phần tử khối 2, chỉ số 4..4
      base=[3, 5, 5, 5, 10, 9]  lazy=[0, 3, 0]  cnt=[{3: 1, 5: 1}, {5: 2}, {9: 1, 10: 1}]

truy vấn 4: [2, 6]
   a=1(x1) -> t=5, mỗi khối tra cnt.get(t - lazy): [1, 0, 0]
   a=3(x1) -> t=3, mỗi kh

## 9. Code nộp LeetCode

`class Solution` là Cách 2 (sqrt decomposition thuần Python). `SolutionNumpy` là Cách 3 dự phòng.

In [12]:
from typing import List
from collections import Counter
import math

class Solution:
    def numberOfPairs(self, nums1: list[int], nums2: list[int], queries: list[list[int]]) -> list[int]:
        n = len(nums2)
        B = max(1, int(math.sqrt(5 * n)))
        nb = (n + B - 1) // B
        base = list(nums2)
        lazy = [0] * nb
        cnt = [Counter(base[b * B:(b + 1) * B]) for b in range(nb)]
        groups = list(Counter(nums1).items())

        def apply_partial(b, l, r, val):
            c = cnt[b]
            for i in range(l, r + 1):
                old = base[i]
                c[old] -= 1
                new = old + val
                base[i] = new
                c[new] += 1

        out = []
        for q in queries:
            if q[0] == 1:
                _, x, y, val = q
                bx, by = x // B, y // B
                if bx == by:
                    apply_partial(bx, x, y, val)
                else:
                    apply_partial(bx, x, (bx + 1) * B - 1, val)
                    for b in range(bx + 1, by):
                        lazy[b] += val
                    apply_partial(by, by * B, y, val)
            else:
                tot = q[1]
                ans = 0
                for a, mult in groups:
                    t = tot - a
                    s = 0
                    for c, lz in zip(cnt, lazy):
                        s += c.get(t - lz, 0)
                    ans += mult * s
                out.append(ans)
        return out


class SolutionNumpy:
    def numberOfPairs(self, nums1: list[int], nums2: list[int], queries: list[list[int]]) -> list[int]:
        import numpy as np
        arr = np.array(nums2, dtype=np.int64)
        targets = list(nums1)
        out = []
        for q in queries:
            if q[0] == 1:
                arr[q[1]:q[2] + 1] += q[3]
            else:
                tot = q[1]
                out.append(sum(int(np.count_nonzero(arr == tot - a)) for a in targets))
        return out

for args, exp in EXAMPLES:
    assert Solution().numberOfPairs(*args) == exp
    assert SolutionNumpy().numberOfPairs(*args) == exp
print("2 lớp đều khớp 3 ví dụ")

2 lớp đều khớp 3 ví dụ


## 10. Vì sao thuật toán đúng

**Bất biến.** Với mọi chỉ số `i` thuộc khối `b`: giá trị thật của `nums2[i]` bằng `base[i] + lazy[b]`; và `cnt[b]` là bảng đếm chính xác của `{base[i] : i ∈ khối b}`.

- **Khởi tạo:** `lazy = 0`, `cnt` dựng từ `base`. ✓
- **Cộng khối đầy:** mọi phần tử thật của khối tăng `val`. Tăng `lazy[b]` đúng bằng ấy mà `base` không đổi, nên `cnt[b]` vẫn đúng. ✓
- **Cộng phần tử lẻ:** với mỗi phần tử bị phủ, `base[i] += val` làm giá trị thật tăng `val` (vì `lazy[b]` giữ nguyên); `cnt` cập nhật (`−1` ở `old`, `+1` ở `old + val`). ✓ (Trường hợp `x` và `y` cùng khối xử lý đúng một lần.)
- **Đếm:** số phần tử thật bằng `t` trong khối `b` là số `i` với `base[i] + lazy[b] = t`, tức `cnt[b][t − lazy[b]]`. Tổng qua các khối cho tổng số phần tử bằng `t`; nhân với số lần xuất hiện của `a` trong `nums1` và cộng lại cho số cặp `(j, k)`. ✓

Các phần tử của khối đầu/cuối được bao phủ **một lần** vì ta tách rõ `bx`, các khối giữa, `by`.

Cách 3 đúng hiển nhiên vì cài đặt đúng theo định nghĩa.

## 11. Phân tích Time/Space

Đặt `n = len(nums2)`, `q = len(queries)`, `A = len(nums1) ≤ 5`, `B` kích thước khối, `nb = ⌈n/B⌉`.

| Bước | Thời gian |
|---|---|
| Khởi tạo bảng đếm | `O(n)` |
| Truy vấn cộng | `O(B + nb)` (hai khối lẻ + `nb` khối giữa) |
| Truy vấn đếm | `O(A · nb)` (mỗi khối một lần tra `dict`) |

Chọn `B ≈ √(A·n)`: mỗi truy vấn `O(√(A·n))` ⇒ **tổng `O(n + q·√(A·n))`**. Với `n = q = 5·10^4`, `A = 5`: khoảng `5·10^4 · 500 = 2.5·10^7` phép `dict`/vòng lặp (thực đo ~ vài giây đến chục giây trong Python).

Không gian `O(n)` (mảng `base` + các bảng đếm; mỗi bảng không quá `B` khóa, nhưng khóa cũ có thể còn `0` — không ảnh hưởng đúng đắn, chỉ giữ thêm bộ nhớ; nếu muốn có thể `del` khi về 0).

Cách 3: `O(q · n · A)` thao tác vector hóa, không gian `O(n)`.

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, cộng toàn mảng, `nums1` có phần tử trùng, `tot` quá lớn), phản ví dụ, và đối chiếu ngẫu nhiên với vét cạn với nhiều kích thước khối.

In [13]:
sol = Solution(); solnp = SolutionNumpy()
for args, exp in EXAMPLES:
    assert sol.numberOfPairs(*args) == exp
    assert solnp.numberOfPairs(*args) == exp

assert sol.numberOfPairs([1], [1], [[2, 2]]) == [1]                          # n = 1
assert sol.numberOfPairs([1], [1], [[1, 0, 0, 5], [2, 7], [2, 2]]) == [1, 0]
assert sol.numberOfPairs([3, 3, 3], [1, 1], [[2, 4]]) == [6]                  # nums1 có phần tử trùng: 3 * 2 cặp
assert sol.numberOfPairs([5], [5, 5, 5, 5], [[2, 10], [1, 0, 3, 1], [2, 10], [2, 11]]) == [4, 0, 4]
assert sol.numberOfPairs([1], [1, 2], [[2, 10**9]]) == [0]                    # tot lớn: không có cặp
print("ví dụ + edge case OK")

ví dụ + edge case OK


In [14]:
def gen_case(rng, nmax, qmax, vmax, valmax):
    n = rng.randint(1, nmax)
    nums1 = [rng.randint(1, vmax) for _ in range(rng.randint(1, 5))]
    nums2 = [rng.randint(1, vmax) for _ in range(n)]
    queries = []
    for _ in range(rng.randint(1, qmax)):
        if rng.random() < 0.5:
            x = rng.randrange(n); y = rng.randrange(x, n)
            queries.append([1, x, y, rng.randint(1, valmax)])
        else:
            queries.append([2, rng.randint(1, 2 * vmax + valmax * 3)])
    return nums1, nums2, queries

rng = random.Random(3943)
for _ in range(400):
    nums1, nums2, queries = gen_case(rng, 15, 30, 6, 3)
    exp = brute(nums1, nums2, queries)
    for B in (1, 2, 3, 5, 7):
        assert sqrt_decomp(nums1, nums2, queries, B) == exp, (nums1, nums2, queries, B)
    assert numpy_brute(nums1, nums2, queries) == exp
    assert sol.numberOfPairs(nums1, nums2, queries) == exp
    assert solnp.numberOfPairs(nums1, nums2, queries) == exp
print("400 test nhỏ: brute = sqrt (nhiều B) = numpy = Solution")

for _ in range(60):
    nums1, nums2, queries = gen_case(rng, 300, 300, 40, 10)
    exp = brute(nums1, nums2, queries)
    assert sol.numberOfPairs(nums1, nums2, queries) == exp
    assert solnp.numberOfPairs(nums1, nums2, queries) == exp
print("60 test vừa khớp")

400 test nhỏ: brute = sqrt (nhiều B) = numpy = Solution


60 test vừa khớp


In [15]:
# Thời gian
rng = random.Random(1)
def big_case(n, q):
    nums1 = [rng.randint(1, 10**5) for _ in range(5)]
    nums2 = [rng.randint(1, 10**5) for _ in range(n)]
    queries = []
    for _ in range(q):
        if rng.random() < 0.5:
            x = rng.randrange(n); y = rng.randrange(x, n)
            queries.append([1, x, y, rng.randint(1, 10**5)])
        else:
            queries.append([2, rng.randint(1, 10**9)])
    return nums1, nums2, queries

args = big_case(50000, 50000)
t0 = time.time(); r2 = SolutionNumpy().numberOfPairs(*args); print(f"numpy,  n = q = 5e4: {time.time() - t0:.2f}s")
args_small = big_case(15000, 15000)
t0 = time.time(); r1 = Solution().numberOfPairs(*args_small); t1 = time.time() - t0
r1b = SolutionNumpy().numberOfPairs(*args_small)
assert r1 == r1b
print(f"sqrt,   n = q = 1.5e4: {t1:.2f}s (tăng ~ q·√n; ở 5e4 chậm gấp ~6 lần)")

numpy,  n = q = 5e4: 1.59s


sqrt,   n = q = 1.5e4: 1.49s (tăng ~ q·√n; ở 5e4 chậm gấp ~6 lần)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tra `cnt.get(t)` mà quên trừ `lazy`

**Code sai:** `s += c.get(t, 0)`.
**Vì sao sai:** `cnt` lưu `base`, giá trị thật là `base + lazy`.
**Code đúng:** `c.get(t - lz, 0)`.

In [16]:
def wrong_forget_lazy(nums1, nums2, queries, B=2):
    n = len(nums2); nb = (n + B - 1) // B
    base = list(nums2); lazy = [0] * nb
    cnt = [Counter(base[b * B:(b + 1) * B]) for b in range(nb)]
    out = []
    for q in queries:
        if q[0] == 1:
            _, x, y, val = q
            bx, by = x // B, y // B
            def part(b, l, r):
                c = cnt[b]
                for i in range(l, r + 1):
                    old = base[i]; c[old] -= 1; base[i] = old + val; c[old + val] += 1
            if bx == by:
                part(bx, x, y)
            else:
                part(bx, x, (bx + 1) * B - 1)
                for b in range(bx + 1, by):
                    lazy[b] += val
                part(by, by * B, y)
        else:
            tot = q[1]
            out.append(sum(sum(c.get(tot - a, 0) for c in cnt) for a in nums1))   # SAI: thiếu - lazy
    return out

args = ([1, 1], [2, 2, 3, 3, 4, 4], [[1, 0, 5, 1], [2, 4]])
print("sai :", wrong_forget_lazy(*args), " đúng:", brute(*args))

sai : [8]  đúng: [4]


### Lỗi 2 — Cộng khối đầy bằng cách duyệt từng phần tử (TLE) hoặc quên cập nhật bảng đếm ở khối lẻ

**Code sai:** ở khối lẻ chỉ sửa `base[i]` mà không đổi `cnt`.
**Vì sao sai:** bảng đếm lệch với `base`, truy vấn sau cho kết quả sai.
**Code đúng:** `cnt[b][old] -= 1; cnt[b][new] += 1`.

In [17]:
def wrong_stale_counter(nums1, nums2, queries, B=2):
    n = len(nums2); nb = (n + B - 1) // B
    base = list(nums2); lazy = [0] * nb
    cnt = [Counter(base[b * B:(b + 1) * B]) for b in range(nb)]
    out = []
    for q in queries:
        if q[0] == 1:
            _, x, y, val = q
            bx, by = x // B, y // B
            for i in range(x, y + 1):
                if i // B in (bx, by):
                    base[i] += val                # SAI: không cập nhật cnt
                elif (i // B) > bx and (i // B) < by and i % B == 0:
                    lazy[i // B] += val
        else:
            tot = q[1]
            out.append(sum(sum(c.get(tot - a - lz, 0) for c, lz in zip(cnt, lazy)) for a in nums1))
    return out

args = ([1], [2, 2], [[1, 0, 0, 3], [2, 6]])
print("sai :", wrong_stale_counter(*args), " đúng:", brute(*args))

sai : [0]  đúng: [1]


### Lỗi 3 — Cộng hai lần khi `x` và `y` cùng một khối

**Code sai:** luôn gọi `apply_partial(bx, x, cuối khối bx)` rồi `apply_partial(by, đầu khối by, y)`.
**Vì sao sai:** nếu `bx == by` thì vừa cộng cho `x..hết khối` (có thể vượt quá `y`) vừa cộng `đầu khối..y` (chồng với đoạn trước).
**Code đúng:** tách trường hợp `bx == by`.

In [18]:
def wrong_same_block(nums1, nums2, queries, B=4):
    n = len(nums2); nb = (n + B - 1) // B
    base = list(nums2); lazy = [0] * nb
    cnt = [Counter(base[b * B:(b + 1) * B]) for b in range(nb)]
    out = []
    for q in queries:
        if q[0] == 1:
            _, x, y, val = q
            bx, by = x // B, y // B
            def part(b, l, r):
                c = cnt[b]
                for i in range(l, min(r, n - 1) + 1):
                    old = base[i]; c[old] -= 1; base[i] = old + val; c[old + val] += 1
            part(bx, x, (bx + 1) * B - 1)                 # SAI khi bx == by
            for b in range(bx + 1, by):
                lazy[b] += val
            part(by, by * B, y)
        else:
            tot = q[1]
            out.append(sum(sum(c.get(tot - a - lz, 0) for c, lz in zip(cnt, lazy)) for a in nums1))
    return out

args = ([1], [1, 1, 1, 1], [[1, 1, 1, 2], [2, 4]])
print("sai :", wrong_same_block(*args), " đúng:", brute(*args))

sai : [3]  đúng: [1]


### Lỗi 4 — Bỏ qua phần tử trùng trong `nums1`

**Code sai:** dùng `set(nums1)`.
**Vì sao sai:** đề đếm **cặp chỉ số `(j, k)`**; hai phần tử `nums1` bằng nhau cho hai tập cặp khác nhau.
**Code đúng:** nhân với số lần xuất hiện (`Counter(nums1)`).

In [19]:
def wrong_set(nums1, nums2, queries):
    arr = list(nums2); out = []
    for q in queries:
        if q[0] == 1:
            for i in range(q[1], q[2] + 1):
                arr[i] += q[3]
        else:
            c = Counter(arr)
            out.append(sum(c.get(q[1] - a, 0) for a in set(nums1)))       # SAI: set
    return out

args = ([1, 1], [2, 2, 3], [[2, 4]])
print("sai :", wrong_set(*args), " đúng:", brute(*args))

sai : [1]  đúng: [2]


### Lỗi 5 — Dựng `Counter(nums2)` mới mỗi lần có truy vấn loại 2

Mỗi truy vấn tốn `O(n)`: `5·10^4 · 5·10^4 = 2.5·10^9` (xem Cách 1). Hãy giữ sẵn bảng đếm theo khối.

## 14. Mẹo & nhận dạng pattern

- **"Cộng cả đoạn + đếm số phần tử bằng một giá trị"** là bài "khó" cho segment tree thuần (không gộp được). Dấu hiệu quen thuộc cho **phân rã căn bậc hai** với **bảng đếm theo khối + tag lười**.
- Khi một mảng rất ngắn (≤ 5) xuất hiện, hãy để vòng lặp qua nó **bên ngoài** và biến mỗi phần tử thành một câu hỏi đơn giản trên mảng dài.
- Với Python, đừng bỏ qua **NumPy**: vét cạn vector hóa có thể thắng thuật toán tốt hơn về tiệm cận nhờ hằng số.
- Khi chia khối, chọn `B` bằng cách **cân bằng chi phí hai loại truy vấn**: cộng tốn `B`, đếm tốn `A·n/B` → `B ≈ √(A·n)`.
- Giá trị trong khối được lưu dưới dạng `base` (không gồm `lazy`) để cộng khối đầy là `O(1)`.

## 15. Biến thể

1. **`val` có thể âm:** vẫn đúng (không dùng tính đơn điệu).
2. **Đếm cặp có tổng `≥ tot`:** cần cấu trúc đếm theo giá trị `≥ t` trong khối (sort từng khối + `bisect`, khi sửa lẻ thì sắp xếp lại khối: `O(B log B)`).
3. **Hai mảng đều bị cộng đoạn:** không còn "mảng ngắn" nên cần kỹ thuật khác (ví dụ chia thời gian thành khối, xử lý ngoại tuyến).
4. **Gán đoạn (set) thay vì cộng:** tag lười thay bằng "gán" — bảng đếm khối chỉ còn một khóa.
5. **Truy vấn đếm phần tử bằng `t` trong đoạn `[l, r]`:** hai khối lẻ duyệt từng phần tử, các khối giữa tra `cnt` — cùng khung với bài này.

In [20]:
# Biến thể 5: đếm số phần tử thật bằng t trong đoạn [l, r], dùng cùng cấu trúc khối
class RangeAddCount:
    def __init__(self, arr, B=4):
        self.n = len(arr); self.B = B
        self.nb = (self.n + B - 1) // B
        self.base = list(arr); self.lazy = [0] * self.nb
        self.cnt = [Counter(self.base[b * B:(b + 1) * B]) for b in range(self.nb)]
    def add(self, x, y, val):
        B = self.B
        for b in range(x // B, y // B + 1):
            lo, hi = max(x, b * B), min(y, (b + 1) * B - 1)
            if lo == b * B and hi == min(self.n, (b + 1) * B) - 1:
                self.lazy[b] += val
            else:
                for i in range(lo, hi + 1):
                    self.cnt[b][self.base[i]] -= 1; self.base[i] += val; self.cnt[b][self.base[i]] += 1
    def count_equal(self, l, r, t):
        B, res = self.B, 0
        for b in range(l // B, r // B + 1):
            lo, hi = max(l, b * B), min(r, (b + 1) * B - 1)
            if lo == b * B and hi == min(self.n, (b + 1) * B) - 1:
                res += self.cnt[b].get(t - self.lazy[b], 0)
            else:
                res += sum(1 for i in range(lo, hi + 1) if self.base[i] + self.lazy[b] == t)
        return res

ds = RangeAddCount([1, 2, 2, 3, 2, 2, 1, 2, 5], B=3)
ds.add(1, 6, 1)               # [1,3,3,4,3,3,2,2,5]
real = [1, 3, 3, 4, 3, 3, 2, 2, 5]
for l, r, t in [(0, 8, 3), (2, 5, 3), (4, 8, 2), (3, 3, 4)]:
    assert ds.count_equal(l, r, t) == sum(1 for v in real[l:r + 1] if v == t)
print("RangeAddCount OK")

RangeAddCount OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute Force | cộng từng phần tử, dựng `Counter` mỗi truy vấn | `O(q · n)` | `O(n)` | kiểm chứng, `n·q ≲ 10^6` |
| 2. Sqrt decomposition | khối + `lazy` + `Counter` theo khối | `O(n + q·√(A·n))` | `O(n)` | thuật toán chuẩn, không cần NumPy |
| 3. NumPy vét cạn | `arr[x:y+1] += v`, `count_nonzero(arr == t)` | `O(q·n·A)` (hằng số nhỏ) | `O(n)` | thực tế nhanh nhất trong Python nếu có NumPy |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `count_pairs_once([2, 2], [1, 1, 3], 3)` bằng bao nhiêu? (Cẩn thận: `nums1` có hai số `2`.)

**Bài 2 (tính tay).** `nums1 = [1]`, `nums2 = [4, 4, 4]`, truy vấn `[1,0,1,2]`, rồi `[2,6]`, rồi `[2,5]`. Hai đáp án là gì?

**Bài 3 (điền TODO).** Hoàn thiện `my_block_range_add(base, lazy, cnt, B, x, y, val)` cộng đoạn vào cấu trúc khối (khối đầy dùng `lazy`).

**Bài 4 (tự viết lại).** Viết `my_number_of_pairs(nums1, nums2, queries)` theo Cách 2, không nhìn code nộp.

**Bài 5 (biến thể).** Viết `count_pairs_sum_at_least(nums1, nums2, queries)` — truy vấn `[2, tot]` đếm cặp `nums1[j] + nums2[k] ≥ tot` (vét cạn đủ cho test nhỏ).

In [21]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_block_range_add(base, lazy, cnt, B, x, y, val):
    # TODO 1: bx, by = x // B, y // B
    # TODO 2: nếu bx == by: sửa từng phần tử x..y (cnt[b][old] -= 1; base[i] += val; cnt[b][new] += 1)
    # TODO 3: ngược lại sửa phần lẻ khối bx, cộng lazy cho khối giữa, sửa phần lẻ khối by
    pass

# Bài 4 — TODO
def my_number_of_pairs(nums1, nums2, queries):
    # TODO: B = int(sqrt(5n)), base/lazy/cnt, gom nums1 thành (a, mult), xử lý từng truy vấn
    pass

# Bài 5 — TODO
def count_pairs_sum_at_least(nums1, nums2, queries):
    # TODO: vét cạn: cộng đoạn và đếm số cặp có tổng >= tot
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `nums1 = [2,2]` hai chỉ số, cần `nums2 == 1`.
- Mức 2: `nums2` có hai số `1`.
- Mức 3: mỗi `2` ghép với 2 số `1` → `2 · 2 = 4`.

**Bài 2.**
- Mức 1: sau `[1,0,1,2]`: `nums2 = [6, 6, 4]`.
- Mức 2: `[2,6]` cần `nums2 == 5`; `[2,5]` cần `nums2 == 4`.
- Mức 3: `[0, 1]` (không có `5`; có một `4`).

**Bài 3.**
- Mức 1: tách ba trường hợp `bx == by`, hoặc không.
- Mức 2: khối đầy ở giữa chỉ `lazy[b] += val`.
- Mức 3: khối lẻ phải sửa cả `base` lẫn `cnt`.

**Bài 4.**
- Mức 1: xem lại hàm `sqrt_decomp`.
- Mức 2: tra `c.get(t - lz, 0)`.
- Mức 3: nhớ nhân `mult` cho `a` trùng.

**Bài 5.**
- Mức 1: nếu vét cạn, cộng thẳng vào danh sách.
- Mức 2: đếm hai vòng lặp `for a in nums1: for v in arr`.
- Mức 3: điều kiện `a + v >= tot`.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert count_pairs_once([2, 2], [1, 1, 3], 3) == 4
# Bài 2
assert brute([1], [4, 4, 4], [[1, 0, 1, 2], [2, 6], [2, 5]]) == [0, 1]
assert Solution().numberOfPairs([1], [4, 4, 4], [[1, 0, 1, 2], [2, 6], [2, 5]]) == [0, 1]

# Bài 3
def my_block_range_add_ans(base, lazy, cnt, B, x, y, val):
    def part(b, l, r):
        for i in range(l, r + 1):
            old = base[i]
            cnt[b][old] -= 1
            base[i] = old + val
            cnt[b][old + val] += 1
    bx, by = x // B, y // B
    if bx == by:
        part(bx, x, y)
    else:
        part(bx, x, (bx + 1) * B - 1)
        for b in range(bx + 1, by):
            lazy[b] += val
        part(by, by * B, y)

real = [3, 1, 4, 1, 5, 9, 2, 6]
B = 3
base = list(real); lazy = [0] * 3
cnt = [Counter(base[b * B:(b + 1) * B]) for b in range(3)]
my_block_range_add_ans(base, lazy, cnt, B, 1, 6, 2)
for i in range(1, 7):
    real[i] += 2
assert [base[i] + lazy[i // B] for i in range(8)] == real
assert all(cnt[b][v] == sum(1 for i in range(b * B, min(8, (b + 1) * B)) if base[i] == v) for b in range(3) for v in set(base))
print("Bài 1–3 OK")

Bài 1–3 OK


In [23]:
# Bài 4
def my_number_of_pairs_ans(nums1, nums2, queries):
    n = len(nums2)
    B = max(1, int(math.sqrt(5 * n)))
    nb = (n + B - 1) // B
    base = list(nums2); lazy = [0] * nb
    cnt = [Counter(base[b * B:(b + 1) * B]) for b in range(nb)]
    groups = list(Counter(nums1).items())
    out = []
    for q in queries:
        if q[0] == 1:
            my_block_range_add_ans(base, lazy, cnt, B, q[1], q[2], q[3])
        else:
            out.append(sum(m * sum(c.get(q[1] - a - lz, 0) for c, lz in zip(cnt, lazy)) for a, m in groups))
    return out

rng = random.Random(21)
for _ in range(100):
    nums1, nums2, queries = gen_case(rng, 25, 30, 6, 3)
    assert my_number_of_pairs_ans(nums1, nums2, queries) == brute(nums1, nums2, queries)

# Bài 5
def count_pairs_sum_at_least_ans(nums1, nums2, queries):
    arr = list(nums2); out = []
    for q in queries:
        if q[0] == 1:
            for i in range(q[1], q[2] + 1):
                arr[i] += q[3]
        else:
            out.append(sum(1 for a in nums1 for v in arr if a + v >= q[1]))
    return out

assert count_pairs_sum_at_least_ans([1, 2], [3, 4], [[2, 5], [1, 0, 0, 2], [2, 7]]) == [3, 1]
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (khối + tag lười + bảng đếm):**

```python
B = int(sqrt(A * n)); nb = ceil(n / B)
base = nums2[:]; lazy = [0] * nb
cnt = [Counter(base[b*B:(b+1)*B]) for b in range(nb)]
cộng [x, y, val]:
    bx, by = x // B, y // B
    nếu bx == by: sửa lẻ x..y
    ngược lại: sửa lẻ khối bx, lazy += val cho khối giữa, sửa lẻ khối by
    sửa lẻ: cnt[b][old] -= 1; base[i] += val; cnt[b][base[i]] += 1
đếm tot:
    với mỗi (a, mult) trong Counter(nums1): sum(c.get(tot - a - lz, 0) for c, lz in zip(cnt, lazy)) * mult
```

**Checklist trước khi submit:**

- [ ] Tra `c.get(t - lazy, 0)`, không phải `c.get(t, 0)`.
- [ ] Khối đầy chỉ cộng `lazy`; khối lẻ sửa cả `base` và `cnt`.
- [ ] `bx == by` xử lý **một lần**.
- [ ] Nhân số lần xuất hiện của mỗi `a` trong `nums1` (đếm theo chỉ số).
- [ ] `B ≈ √(5n)`, `B ≥ 1`; khối cuối có thể ngắn.
- [ ] Đối chiếu vét cạn trên input ngẫu nhiên với nhiều `B` (kể cả `B = 1`).
- [ ] Chạy thử `n = q = 5·10^4`; nếu Python quá chậm dùng phương án NumPy.